# Overall Performance Analysis

This notebook loads completed experiment reports for a configurable model, converts the report JSONs and query JSONLs into DataFrames, and computes overall model performance across runs.

Metrics included: TCR, Pass@3, inter-run consistency, PSV, CCR, tool appropriateness, latency, tokens per task, tokens per successful task, and cost per successful task.

## Metric Definitions

- **TCR**: task completion rate, using evaluated task pass/fail outcomes across all selected runs.
- **Pass@3**: share of unique tasks that passed in at least one of the three runs.
- **Inter-run consistency**: mean per-task majority agreement across runs. A task passed in all three or failed in all three has consistency `1.0`; a 2-vs-1 split has consistency `0.667`.
- **Average PSV**: mean parameter schema valid rate across task-runs.
- **Average CCR**: mean constraint compliance rate across task-runs, excluding tasks where CCR is not applicable.
- **Average tool appropriateness**: mean task-level tool appropriateness score from the full report.
- **Average latency**: mean task latency in seconds.
- **Average tokens per task**: mean total tokens across task-runs.
- **Tokens per successful task**: total tokens spent divided by number of successful task-runs.
- **Cost per successful task**: total estimated API cost divided by number of successful task-runs. Edit the pricing constants below if your provider/rate differs.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs pandas. Install it with: uv add pandas numpy, "
        "or pip install pandas numpy in the notebook kernel environment."
    ) from exc

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [5]:
# Model/configuration
# Change these values when reusing the notebook for another model.
#MODEL_DISPLAY_NAME = "Qwen3.6 Plus"
MODEL_DISPLAY_NAME = "Kimi K2.6"
#MODEL_LOG_DIR_NAME = "35tasks-qwen3.6-plus"  # folder under logs/
MODEL_LOG_DIR_NAME = "35tasks-kimi-k2.6"  # folder under logs/
MODEL_ID = "qwen/qwen3.6-plus"  # optional, used for labels/pricing notes only

# Leave as None to derive a file-safe prefix from MODEL_LOG_DIR_NAME.
MODEL_EXPORT_PREFIX = None

# Pricing defaults for the selected model. Change these if your provider/rate differs.
INPUT_USD_PER_1M = 0.325
OUTPUT_USD_PER_1M = 1.95

# Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else Path(".").resolve()
LOG_ROOT = PROJECT_ROOT / "logs" / MODEL_LOG_DIR_NAME
EXPORT_PREFIX = MODEL_EXPORT_PREFIX or re.sub(r"[^A-Za-z0-9]+", "_", MODEL_LOG_DIR_NAME).strip("_").lower()

# Use the three numbered production runs and ignore pilot runs.
RUN_DIR_PATTERN = re.compile(r"^(?P<run_index>\d+).*$")

assert LOG_ROOT.exists(), f"Log root not found: {LOG_ROOT}"

In [6]:
def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def selected_run_dirs(log_root: Path) -> list[Path]:
    dirs = []
    for path in log_root.iterdir():
        if not path.is_dir():
            continue
        match = RUN_DIR_PATTERN.match(path.name)
        if match and (path / "task_run_report.json").exists():
            dirs.append(path)
    return sorted(dirs, key=lambda p: int(RUN_DIR_PATTERN.match(p.name).group("run_index")))


run_dirs = selected_run_dirs(LOG_ROOT)
run_dirs

[WindowsPath('d:/projects/thesis-code/logs/35tasks-kimi-k2.6/1moonshotai_kimi-k2.6_20260512T065725Z'),
 WindowsPath('d:/projects/thesis-code/logs/35tasks-kimi-k2.6/2moonshotai_kimi-k2.6_20260512T071213Z'),
 WindowsPath('d:/projects/thesis-code/logs/35tasks-kimi-k2.6/3moonshotai_kimi-k2.6_20260512T071245Z')]

In [7]:
report_rows = []
task_rows = []
query_rows = []

for run_dir in run_dirs:
    match = RUN_DIR_PATTERN.match(run_dir.name)
    run_index = int(match.group("run_index"))
    run_id = run_dir.name

    report = read_json(run_dir / "task_run_report.json")
    metadata = read_json(run_dir / "run_metadata.json") if (run_dir / "run_metadata.json").exists() else {}
    summary = report.get("summary", {})

    report_rows.append({
        "run_index": run_index,
        "run_id": run_id,
        "generated_at": report.get("generated_at"),
        "source_csv": report.get("source_csv"),
        "task_set_id": metadata.get("task_set_id"),
        "model": summary.get("model") or metadata.get("tool_agent_model"),
        **summary,
    })

    for result in report.get("results", []):
        validation = result.get("ground_truth_validation") or {}
        tool_app = result.get("tool_appropriateness") or {}
        query_trace = result.get("query_trace") or {}

        task_rows.append({
            "run_index": run_index,
            "run_id": run_id,
            "task_id": result.get("task_id"),
            "prompt": result.get("prompt"),
            "status": result.get("status"),
            "answer_type": result.get("answer_type"),
            "passed": validation.get("passed", result.get("passed")),
            "ground_truth_score": validation.get("score"),
            "parameter_schema_valid_rate": result.get("parameter_schema_valid_rate"),
            "constraint_compliance_rate": result.get("constraint_compliance_rate"),
            "tool_appropriateness": tool_app.get("average_score"),
            "total_tokens": result.get("total_tokens"),
            "input_tokens": query_trace.get("input_tokens"),
            "output_tokens": query_trace.get("output_tokens"),
            "total_steps": result.get("total_steps"),
            "total_tool_calls": result.get("total_tool_calls"),
            "latency": result.get("latency"),
            "end_to_end_latency": result.get("end_to_end_latency"),
            "attempt_count": result.get("attempt_count"),
            "tools_used": result.get("tools_used") or query_trace.get("tools_used"),
            "final_answer": query_trace.get("final_answer", result.get("final_answer")),
        })

    for row in read_jsonl(run_dir / "query_runs.jsonl"):
        query_rows.append({"run_index": run_index, "run_id": run_id, **row})

reports_df = pd.DataFrame(report_rows)
tasks_df = pd.DataFrame(task_rows)
query_runs_df = pd.DataFrame(query_rows)

tasks_df["passed"] = tasks_df["passed"].astype("boolean")
for col in ["total_tokens", "input_tokens", "output_tokens", "latency", "end_to_end_latency", "parameter_schema_valid_rate", "constraint_compliance_rate", "tool_appropriateness"]:
    tasks_df[col] = pd.to_numeric(tasks_df[col], errors="coerce")

tasks_df["estimated_cost_usd"] = (
    tasks_df["input_tokens"].fillna(0) / 1_000_000 * INPUT_USD_PER_1M
    + tasks_df["output_tokens"].fillna(0) / 1_000_000 * OUTPUT_USD_PER_1M
)

reports_df

,run_index,run_id,generated_at,source_csv,task_set_id,model,total_tasks,finished_with_result_tasks,failed_tasks,tasks_retried,tasks_with_auth_refresh,tasks_with_cleanup_warning,Evaluated tasks,Task Completion Rate (TCR),average_tool_appropriateness,macro_average_parameter_schema_valid_rate(PSV),macro_average_constraint_compliance_rates(CCR),average_total_tokens,judge_token_usage,average_judge_total_tokens,average_total_steps(temp),average_total_tool_calls,average_latency,average_end_to_end_latency,average_attempt_count,jwt_login_count,jwt_refresh_count
0,1,1moonshotai_kimi-k2.6_20260512T065725Z,2026-05-12T07:09:53Z,task\taskset35.csv,taskset_20260512T065725Z_7ff386,moonshotai/kimi-k2.6,35,33,2,0,0,0,35,0.8857,0.8499,1.0000,0.9167,"22,019.1818","{'input_tokens': 25172, 'output_tokens': 26787...","2,361.7727",4.3030,6.6970,75.3714,82.7133,1.0000,1,0
1,2,2moonshotai_kimi-k2.6_20260512T071213Z,2026-05-12T07:35:34Z,task\taskset35.csv,taskset_20260512T071213Z_00036b,moonshotai/kimi-k2.6,35,35,0,0,0,0,35,0.8571,0.8353,1.0000,0.9087,"23,319.3714","{'input_tokens': 26005, 'output_tokens': 26215...","2,373.6364",4.1714,7.0571,95.0767,102.3990,1.0000,1,0
2,3,3moonshotai_kimi-k2.6_20260512T071245Z,2026-05-12T07:25:36Z,task\taskset35.csv,taskset_20260512T071245Z_87c558,moonshotai/kimi-k2.6,35,35,0,0,0,0,35,0.8286,0.8547,1.0000,0.9095,"20,761.8286","{'input_tokens': 25746, 'output_tokens': 25897...","2,347.4091",3.9143,7.6571,76.1153,83.2988,1.0000,1,0


## Loaded DataFrames

In [8]:
print(f"Reports: {reports_df.shape[0]} runs x {reports_df.shape[1]} columns")
print(f"Task results: {tasks_df.shape[0]} task-runs x {tasks_df.shape[1]} columns")
print(f"Query runs: {query_runs_df.shape[0]} rows x {query_runs_df.shape[1]} columns")

display(tasks_df.head())
display(query_runs_df.head())

Reports: 3 runs x 27 columns
Task results: 105 task-runs x 22 columns
Query runs: 105 rows x 27 columns


,run_index,run_id,task_id,prompt,status,answer_type,passed,ground_truth_score,parameter_schema_valid_rate,constraint_compliance_rate,tool_appropriateness,total_tokens,input_tokens,output_tokens,total_steps,total_tool_calls,latency,end_to_end_latency,attempt_count,tools_used,final_answer,estimated_cost_usd
0,1,1moonshotai_kimi-k2.6_20260512T065725Z,A1,how many posts does twitter collection have?,completed,scalar,True,1,1.0000,NaN,1.0000,"6,584.0000","6,479.0000",105.0000,2.0000,1.0000,9.3642,9.3750,1,[get_collection_summary],"{""status"":""success"",""answer"":{""value"":1858732...",0.0023
1,1,1moonshotai_kimi-k2.6_20260512T065725Z,A2,Compare the harvesting date span of all availa...,completed,text,True,1,1.0000,NaN,1.0000,"11,255.0000","10,354.0000",901.0000,3.0000,8.0000,38.5680,50.1935,1,"[get_collections, get_collection_summary, get_...","{""status"":""success"",""answer"":{""summary"":""Amon...",0.0051
2,1,1moonshotai_kimi-k2.6_20260512T065725Z,A3,"Compare Reddit, Mastodon, and Twitter by total...",completed,text,True,1,1.0000,NaN,1.0000,"7,176.0000","6,609.0000",567.0000,2.0000,3.0000,26.0681,36.8460,1,"[get_collection_summary, get_collection_summar...","{\n ""status"": ""success"",\n ""answer"": {\n ...",0.0033
3,1,1moonshotai_kimi-k2.6_20260512T065725Z,A4,List all available collections and their start...,completed,text,True,1,1.0000,NaN,1.0000,"10,796.0000","10,246.0000",550.0000,3.0000,8.0000,25.9806,32.1268,1,"[get_collections, get_collection_summary, get_...","{""status"":""success"",""answer"":{""summary"":""Ther...",0.0044
4,1,1moonshotai_kimi-k2.6_20260512T065725Z,A5,Which available collections have harvesting da...,completed,text,True,1,1.0000,NaN,1.0000,"11,040.0000","10,391.0000",649.0000,3.0000,8.0000,25.5347,35.9497,1,"[get_collections, get_collection_summary, get_...","{""status"":""success"",""answer"":{""summary"":""Six ...",0.0046


,run_index,run_id,query_id,user_query,task_set_id,task_id,session_id,model,toolset_id,status,error_type,error_message,start_time,end_time,final_answer,total_steps,total_tool_calls,parameter_schema_valid_rate,ccr_applicable_calls,ccr_true_calls,constraint_compliance_rate,tools_used,input_tokens,output_tokens,total_tokens,metadata,timestamp
0,1,1moonshotai_kimi-k2.6_20260512T065725Z,query_4eec0e4e30ba,how many posts does twitter collection have?,taskset_20260512T065725Z_7ff386,A1,session_e8daa84f34a6,moonshotai/kimi-k2.6,aio_mcp_toolset_v2,completed,NaN,NaN,2026-05-12T06:57:26Z,2026-05-12T06:57:35Z,"{""status"":""success"",""answer"":{""value"":1858732...",2,1,1.0000,0,0,NaN,[get_collection_summary],6479,105,6584,"{'system_prompt_label': 'scalar', 'system_prom...",2026-05-12T06:57:35Z
1,1,1moonshotai_kimi-k2.6_20260512T065725Z,query_eb543e6185e1,List all available collections and their start...,taskset_20260512T065725Z_7ff386,A4,session_1c67ab9c92ce,moonshotai/kimi-k2.6,aio_mcp_toolset_v2,completed,NaN,NaN,2026-05-12T06:57:26Z,2026-05-12T06:57:51Z,"{""status"":""success"",""answer"":{""summary"":""Ther...",3,8,1.0000,0,0,NaN,"[get_collections, get_collection_summary, get_...",10246,550,10796,"{'system_prompt_label': 'text', 'system_prompt...",2026-05-12T06:57:51Z
2,1,1moonshotai_kimi-k2.6_20260512T065725Z,query_9af269fc22ab,"Compare Reddit, Mastodon, and Twitter by total...",taskset_20260512T065725Z_7ff386,A3,session_84959358af4f,moonshotai/kimi-k2.6,aio_mcp_toolset_v2,completed,NaN,NaN,2026-05-12T06:57:26Z,2026-05-12T06:57:51Z,"{\n ""status"": ""success"",\n ""answer"": {\n ...",2,3,1.0000,0,0,NaN,"[get_collection_summary, get_collection_summar...",6609,567,7176,"{'system_prompt_label': 'text', 'system_prompt...",2026-05-12T06:57:51Z
3,1,1moonshotai_kimi-k2.6_20260512T065725Z,query_a16247b73a26,Which available collections have harvesting da...,taskset_20260512T065725Z_7ff386,A5,session_afc22df10e73,moonshotai/kimi-k2.6,aio_mcp_toolset_v2,completed,NaN,NaN,2026-05-12T06:57:35Z,2026-05-12T06:58:00Z,"{""status"":""success"",""answer"":{""summary"":""Six ...",3,8,1.0000,0,0,NaN,"[get_collections, get_collection_summary, get_...",10391,649,11040,"{'system_prompt_label': 'text', 'system_prompt...",2026-05-12T06:58:00Z
4,1,1moonshotai_kimi-k2.6_20260512T065725Z,query_2fccc3b82858,Compare the harvesting date span of all availa...,taskset_20260512T065725Z_7ff386,A2,session_69649c296faa,moonshotai/kimi-k2.6,aio_mcp_toolset_v2,completed,NaN,NaN,2026-05-12T06:57:26Z,2026-05-12T06:58:04Z,"{""status"":""success"",""answer"":{""summary"":""Amon...",3,8,1.0000,0,0,NaN,"[get_collections, get_collection_summary, get_...",10354,901,11255,"{'system_prompt_label': 'text', 'system_prompt...",2026-05-12T06:58:04Z


## Per-Run Summary

In [9]:
per_run_summary = tasks_df.groupby(["run_index", "run_id"], as_index=False).agg(
    evaluated_tasks=("task_id", "nunique"),
    successful_tasks=("passed", "sum"),
    TCR=("passed", "mean"),
    average_PSV=("parameter_schema_valid_rate", "mean"),
    average_CCR=("constraint_compliance_rate", "mean"),
    average_tool_appropriateness=("tool_appropriateness", "mean"),
    average_latency_seconds=("latency", "mean"),
    average_tokens_per_task=("total_tokens", "mean"),
    total_tokens=("total_tokens", "sum"),
    total_cost_usd=("estimated_cost_usd", "sum"),
)

successful_denominator = per_run_summary["successful_tasks"].replace(0, pd.NA)
per_run_summary["tokens_per_successful_task"] = per_run_summary["total_tokens"] / successful_denominator
per_run_summary["cost_per_successful_task_usd"] = per_run_summary["total_cost_usd"] / successful_denominator

display(per_run_summary)

,run_index,run_id,evaluated_tasks,successful_tasks,TCR,average_PSV,average_CCR,average_tool_appropriateness,average_latency_seconds,average_tokens_per_task,total_tokens,total_cost_usd,tokens_per_successful_task,cost_per_successful_task_usd
0,1,1moonshotai_kimi-k2.6_20260512T065725Z,35,31,0.8857,1.0000,0.9167,0.8499,75.3714,"22,019.1818","726,633.0000",0.3444,"23,439.7742",0.0111
1,2,2moonshotai_kimi-k2.6_20260512T071213Z,35,30,0.8571,1.0000,0.9087,0.8353,95.0767,"23,319.3714","816,178.0000",0.4266,"27,205.9333",0.0142
2,3,3moonshotai_kimi-k2.6_20260512T071245Z,35,29,0.8286,1.0000,0.9095,0.8547,76.1153,"20,761.8286","726,664.0000",0.3616,"25,057.3793",0.0125


## Overall Performance Across 3 Runs

In [ ]:
unique_task_count = tasks_df["task_id"].nunique()
successful_task_runs = int(tasks_df["passed"].sum())
total_task_runs = len(tasks_df)

pass_by_task = tasks_df.groupby("task_id")["passed"].agg(
    runs="count",
    passes="sum",
    pass_at_3="any",
)
pass_by_task["fails"] = pass_by_task["runs"] - pass_by_task["passes"]
pass_by_task["majority_agreement"] = pass_by_task[["passes", "fails"]].max(axis=1) / pass_by_task["runs"]
pass_by_task["all_runs_same"] = pass_by_task["majority_agreement"].eq(1.0)

overall_metrics = pd.DataFrame([
    {"Metric": "Task completion rate", "Value": tasks_df["passed"].mean()},
    {"Metric": "Pass@3", "Value": pass_by_task["pass_at_3"].mean()},
    {"Metric": "Inter-run consistency", "Value": pass_by_task["majority_agreement"].mean()},
    {"Metric": "Strict all-run consistency", "Value": pass_by_task["all_runs_same"].mean()},
    {"Metric": "Average Parameter Schema Validity", "Value": tasks_df["parameter_schema_valid_rate"].mean()},
    {"Metric": "Average Constraint Compliance Rate", "Value": tasks_df["constraint_compliance_rate"].mean()},
    {"Metric": "Average Tool Appropriateness", "Value": tasks_df["tool_appropriateness"].mean()},
    {"Metric": "Average latency (s)", "Value": tasks_df["latency"].mean()},
    {"Metric": "Average tokens per task", "Value": tasks_df["total_tokens"].mean()},
    {"Metric": "Tokens per successful task", "Value": tasks_df["total_tokens"].sum() / successful_task_runs},
    {"Metric": "Cost per successful task (USD)", "Value": tasks_df["estimated_cost_usd"].sum() / successful_task_runs},
])

display(overall_metrics)
print(f"Successful task-runs: {successful_task_runs}/{total_task_runs}")
print(f"Unique tasks passed at least once: {int(pass_by_task['pass_at_3'].sum())}/{unique_task_count}")
print(f"Model: {MODEL_DISPLAY_NAME} ({MODEL_ID})")
# print(f"Pricing used: ${INPUT_USD_PER_1M}/1M input tokens, ${OUTPUT_USD_PER_1M}/1M output tokens")

,Metric,Value
0,Task completion rate,0.8571
1,Pass@3,0.9429
2,Inter-run consistency,0.9333
3,Strict all-run consistency,0.8000
4,Average Parameter Schema Validity,1.0000
5,Average Constraint Compliance Rate,0.9116
6,Average Tool Appropriateness,0.8466
7,Average latency (s),82.1878
8,Average tokens per task,"22,033.7379"
9,Tokens per successful task,"25,216.3889"


Successful task-runs: 90/105
Unique tasks passed at least once: 33/35
Model: Kimi K2.6 (qwen/qwen3.6-plus)
Pricing used: $0.325/1M input tokens, $1.95/1M output tokens


## Task-Level Stability

In [11]:
task_stability = pass_by_task.reset_index().sort_values(["majority_agreement", "passes", "task_id"])
display(task_stability)

pass_matrix = tasks_df.pivot_table(index="task_id", columns="run_index", values="passed", aggfunc="first").sort_index()
display(pass_matrix)

,task_id,runs,passes,pass_at_3,fails,majority_agreement,all_runs_same
33,E6,3,1,True,2,0.6667,False
34,E7,3,1,True,2,0.6667,False
12,B6,3,2,True,1,0.6667,False
16,C3,3,2,True,1,0.6667,False
22,D2,3,2,True,1,0.6667,False
23,D3,3,2,True,1,0.6667,False
32,E5,3,2,True,1,0.6667,False
15,C2,3,0,False,3,1.0000,True
24,D4,3,0,False,3,1.0000,True
0,A1,3,3,True,0,1.0000,True


run_index,1,2,3
task_id,,,
A1,True,True,True
A2,True,True,True
A3,True,True,True
A4,True,True,True
A5,True,True,True
A6,True,True,True
A7,True,True,True
B1,True,True,True
B2,True,True,True


## Optional: Export Tables

In [12]:
EXPORT_DIR = PROJECT_ROOT / "analysis" / "outputs"
EXPORT_DIR.mkdir(exist_ok=True)

reports_df.to_csv(EXPORT_DIR / f"{EXPORT_PREFIX}_reports.csv", index=False)
tasks_df.to_csv(EXPORT_DIR / f"{EXPORT_PREFIX}_task_runs.csv", index=False)
query_runs_df.to_csv(EXPORT_DIR / f"{EXPORT_PREFIX}_query_runs.csv", index=False)
per_run_summary.to_csv(EXPORT_DIR / f"{EXPORT_PREFIX}_per_run_summary.csv", index=False)
overall_metrics.to_csv(EXPORT_DIR / f"{EXPORT_PREFIX}_overall_metrics.csv", index=False)
task_stability.to_csv(EXPORT_DIR / f"{EXPORT_PREFIX}_task_stability.csv", index=False)

EXPORT_DIR

WindowsPath('d:/projects/thesis-code/analysis/outputs')